# Bài thực hành 2: GenUI có kiểm soát

Từ đầu khóa học đến giờ, agent của bạn mới chỉ phản hồi lại bằng văn bản thuần túy. Trong bài học này, bạn sẽ nâng cấp agent để nó có thể hiển thị các thành phần giao diện (React UI) phong phú như: thẻ thông tin chuyến bay, biểu đồ tròn, hoặc bất cứ thứ gì bạn lập trình.

Quy trình rất đơn giản: **Bạn xây dựng các component, còn agent sẽ quyết định khi nào nên sử dụng component nào.**

**Mục tiêu bài học:**

- **Hiểu về GenUI:** Nắm bắt khái niệm và hiểu vị trí của GenUI có kiểm soát trong bức tranh tổng thể.
- **Đăng ký các component ở frontend:** Sử dụng hook `useComponent()` để "giới thiệu" các React component cho agent nhận diện.
- **Render đầu ra có cấu trúc:** Cho phép agent tự động chọn và điền dữ liệu vào các component UI ngay trong luồng chat.

**Bạn sẽ xây dựng gì?**

Bạn sẽ tạo ra một giao diện chat có khả năng hiển thị các component UI trực quan dựa trên yêu cầu của người dùng. Ví dụ:

- **Người dùng:** *"Hiển thị thẻ chuyến bay của hãng Pacific Air từ SFO đến JFK khởi hành lúc 08:30 với giá $249"* -> Agent trả về một giao diện thẻ vé máy bay cực kỳ trực quan.

<img src="images/flight-card.png" alt="Flight Card" style="display: block; margin: 0 auto; max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

- **Người dùng:** *"Hãy cho tôi xem phân bổ doanh thu theo danh mục bằng biểu đồ tròn"* -> Agent trả về một biểu đồ tròn tương tác.

<img src="images/pie-chart.png" alt="Pie Chart" style="display: block; margin: 0 auto; max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

## 1. GenUI có kiểm soát là gì?

GenUI là một mô hình nơi AI agent không chỉ trả lời bằng văn bản mà bằng toàn bộ các giao diện tương tác.

GenUI có kiểm soát là biến thể khắt khe và an toàn nhất của mô hình này: Agent **chỉ có thể** hiển thị các component mà bạn đã chủ động đăng ký trước.

### 1.1. Cơ chế hoạt động

Mỗi component bạn đăng ký sẽ được đóng gói thành một "công cụ" cho agent, bao gồm:

- Một **tên** cố định.
- Một **schema dữ liệu đầu vào** được định kiểu rõ ràng.
- Một **React component** được ánh xạ tương ứng.

Agent sẽ không tự ý "chế" ra giao diện. Nó chỉ tạo ra dữ liệu có cấu trúc và truyền vào các component do chính bạn lập trình ở frontend.

### 1.2. Ưu và nhược điểm

**Ưu điểm:**

- **Dễ triển khai:** Chỉ cần đăng ký component là xong.
- **Độ hoàn thiện hình ảnh cao:** Vì mọi giao diện đều do chính tay bạn (lập trình viên) thiết kế và CSS.
- **Độ an toàn tuyệt đối:** Agent chỉ có thể gọi các công cụ đã đăng ký với các tham số đã qua kiểm duyệt.
- **Phù hợp cho môi trường thực tế:** Rất tốt cho các ứng dụng có lượng truy cập cao hoặc hệ thống quan trọng cần sự ổn định.

**Nhược điểm:**

- **Tốn công sức frontend:** Cứ thêm một tính năng/dữ liệu mới, bạn lại phải code thêm một component tương ứng.
- **Thiếu sự tự do:** Kém linh hoạt hơn so với các dạng GenUI mở.

## 2. Hook `useComponent()`

`useComponent()` là một hook của CopilotKit dùng để đăng ký một React component thành một công cụ cho agent gọi bên trong `<CopilotChat />`. Bạn định nghĩa các quy tắc, còn agent sẽ chọn thời điểm sử dụng.

**Cú pháp cơ bản:**

```tsx
useComponent({
  name: "component_name",
  description: "mô tả để agent hiểu component này dùng để làm gì",
  parameters: z.object({ ... }),
  render: MyComponent,
});
```

- `name` (string - bắt buộc): tên công cụ để agent nhận diện.
- `description` (string - tùy chọn): hướng dẫn agent khi nào thì nên gọi công cụ này.
- `parameters` (Zod schema - tùy chọn): cấu trúc dữ liệu sẽ được truyền vào component.
- `render` (bắt buộc): một React component thực thụ, hoặc một hàm nhận vào `{ args, status }` để bạn tự custom luồng render (như thêm trạng thái loading, wrapper,...).

## 3. Thực hành

### 3.1. Chuẩn bị môi trường & Khởi động server

Lưu ý: Đoạn mã Python dưới đây dùng để chạy ngầm backend và frontend trong môi trường học tập.

In [2]:
# Tải các API key cần thiết
from helper import load_api_keys
load_api_keys()

# Khởi động agent backend ở port 8003 (Tương tự agent ở bài thực hành 1)
from backend.server import start_backend
start_backend(port=8003)

# Khởi động Frontend ở port 3003
from helper import start_frontend
start_frontend(port=3003)

✓ OpenAI API key loaded
✓ Google API key loaded
✓ Server running at http://localhost:8003
Starting frontend on port 3003 ...
✓ App running at http://localhost:3003

Read the logs: /home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/2-controlled-generative-ui/frontend/dev-logs.txt


### 3.2. Đăng ký component vào frontend

Chúng ta sẽ chỉnh sửa file `App.tsx` ở phía frontend để đăng ký 3 component bằng `useComponent()`:
- Một thẻ đơn giản hiển thị tên (`showMyName`).
- Một biểu đồ tròn (`pieChart`).
- Một thẻ thông tin chuyến bay (`flightCard`).

In [ ]:
%%writefile frontend/src/App.tsx
import { z } from "zod"
import { CopilotChat } from "@copilotkit/react-core/v2";
import { useComponent } from "@copilotkit/react-core/v2";

import { FlightCard, FlightCardProps } from "@/components/flight-card";
import { PieChart, PieChartProps } from "@/components/pie-chart";

import { useExampleSuggestions } from "@/hooks/use-example-suggestions";

export default function App() {

  // Đăng ký component 1: Hiển thị tên người dùng
  useComponent({
    name: "showMyName",
    description: "Hiển thị tên của người dùng trong một thẻ",
    parameters: z.object({ name: z.string() }),
    render: ({ name }) => <div className="bg-blue-500 p-4">Hi, {name}!</div>,
  });

  // Đăng ký component 2: Biểu đồ tròn hiển thị dữ liệu cấu trúc
  useComponent({
    name: "pieChart",
    description: "Giao diện hiển thị dữ liệu dưới dạng biểu đồ tròn.",
    parameters: PieChartProps,
    render: PieChart,
  });

  // Đăng ký component 3: Thẻ thông tin chuyến bay
  useComponent({
    name: "flightCard",
    description: "Giao diện hiển thị tóm tắt thông tin của một chuyến bay.",
    parameters: FlightCardProps,
    render: FlightCard,
  });

  // Thêm các nút gợi ý câu hỏi vào CopilotChat
  useExampleSuggestions();

  return <CopilotChat />;

};

### 3.3. Trải nghiệm ứng dụng

Bây giờ, agent của bạn đã sở hữu 3 công cụ UI. Hãy mở giao diện và chat với agent bằng các câu lệnh như:

- "Show my name" (Kèm theo tên của bạn).
- "Pie chart" (Yêu cầu vẽ một biểu đồ tỷ lệ).
- "Flight card" (Yêu cầu tìm chuyến bay).

Bạn sẽ thấy thay vì trả lời bằng chữ, agent sẽ render trực tiếp giao diện bóng bẩy ra màn hình chat!

## 4. Đào sâu: Các component thực chất được viết như thế nào?

Bất kỳ React component nào cũng có thể trở thành công cụ cho GenUI - chỉ cần bạn đăng ký nó. Để hiểu rõ hơn, đây là mã nguồn toàn bộ của FlightCard mà chúng ta vừa import ở trên:

```tsx
import { z } from "zod";

export const FlightCardProps = z.object({
  title: z.string().describe("Flight card title"),
  airline: z.string().describe("Airline name"),
  origin: z.string().describe("Departure airport/city"),
  destination: z.string().describe("Arrival airport/city"),
  departure_time: z.string().describe("Departure time"),
  price: z.string().describe("Price display"),
});

type FlightCardProps = z.infer<typeof FlightCardProps>;

export function FlightCard({
  title,
  airline,
  origin,
  destination,
  departure_time,
  price,
}: FlightCardProps) {
  return (
    <div className="rounded-lg border bg-white p-3 space-y-2">
      <div className="font-semibold">{title}</div>
      <div className="rounded border p-2 text-sm">
        <div className="font-medium">{airline}</div>
        <div>
          {origin} → {destination}
        </div>
        <div>Departs: {departure_time}</div>
        <div className="font-semibold">{price}</div>
      </div>
    </div>
  );
}
```

**Tại sao lại dùng Zod?**

[Zod](https://zod.dev) là một thư viện xác thực dữ liệu dành cho TypeScript (rất giống với **Pydantic** trong Python). Zod cho phép bạn định nghĩa cấu trúc props **một lần duy nhất**, và sử dụng cấu trúc đó cho cả việc định nghĩa kiểu trong TypeScript (`type FlightCardProps`), lẫn làm tham số `parameters` truyền vào LLM thông qua `useComponent()`. Điều này đảm bảo agent luôn trả về đúng dữ liệu mà UI cần.

## Tổng kết

- **GenUI có kiểm soát** giới hạn AI chỉ được phép render các component mà lập trình viên đã khai báo tường minh.
- Hook `useComponent()` là cầu nối, định nghĩa một hợp đồng dữ liệu khắt khe (qua Zod) kết nối tham số của AI với Props của React.
- Sự phân chia rạch ròi: Backend và agent phụ trách việc chọn công cụ và tạo dữ liệu, frontend lo việc thiết kế và hiển thị giao diện.
- Phương pháp này hy sinh một chút sự linh hoạt để đổi lấy **trải nghiệm người dùng an toàn, đẹp mắt và có thể dự đoán được 100%**.

**Thử thách cho bạn:** Hãy thử tự tạo một component mới (ví dụ: một bảng hoặc một thanh tiến trình), định nghĩa Zod schema cho nó, đăng ký bằng `useComponent`, và yêu cầu agent sử dụng nó!

## Bước tiếp theo

Trong bài tiếp theo, bạn sẽ tiến tới vùng giữa của phổ GenUI: GenUI khai báo. Thay vì đăng ký từng component riêng lẻ, bạn sẽ định nghĩa một bộ danh mục các "khối xếp hình" và cho phép agent tự do lắp ghép chúng thành các bố cục phức tạp dựa trên tiêu chuẩn A2UI.